# P1. 아파트 실거래가 대시보드

서울 아파트 매매 실거래 넉 달치(2026년 6~9월)를 Iceberg 테이블에 두 번 나눠 적재하고, Spark 로 구별 · 평형별 평당가를 집계해 Streamlit 대시보드로 보여 줍니다. Iceberg 타임트래블로 「어제 기준」 화면도 다시 봅니다.

- 단계: 개별 프로젝트 · 정형 · 예상 2~3일
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 공공데이터포털 — 국토교통부 아파트 매매 실거래가 자료(15126469) OpenAPI, 인증키 필요 · 이용허락 제한 없음
- 데이터: 서울 열린데이터광장 — 서울시 부동산 실거래가 정보(OA-21275) 계약일 2026년 6~9월분(2026-10-28 내려받음), 공공누리 1유형(출처표시)

**빈칸 노트북** — 코드 속 `___①___` 자리 15곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/P1/P1_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

아파트 실거래가는 계약 뒤 30일 안에 신고되고, 국토교통부가 그 신고를 날마다 공개합니다. 서울시는 같은 자료를 이어받아 열린데이터광장에 올립니다.
이 프로젝트에서는 그 자료로 **「어느 구가 평당 얼마인가, 평형에 따라 어떻게 다른가」**를 보여 주는 대시보드를 만듭니다.

데이터가 날마다 늘어나는 대시보드에는 곤란한 질문이 하나 따라옵니다. 「어제 회의 때 본 숫자와 왜 달라요?」
테이블 포맷 Iceberg 는 적재할 때마다 **스냅샷**을 남기므로, 어제 적재가 끝난 시점의 테이블을 그대로 다시 읽을 수 있습니다(타임트래블). 이 기능을 직접 써 봅니다.

```text
OpenAPI(키) 또는 스냅샷 CSV → pandas 정제 → Iceberg 적재 ① (어제)
                                              → Spark 집계 → 마트 테이블 ①
                                   Iceberg 적재 ② (오늘)
                                              → Spark 집계 → 마트 테이블 ②
                                   타임트래블로 ① 다시 읽기 → Streamlit 대시보드
```

다 끝내면 다음을 말할 수 있습니다.

- 서울에서 평당가(3.3㎡당 거래금액)가 가장 높은 구와 가장 낮은 구, 그 차이
- 전용면적 60㎡ 이하 · 60~85㎡ · 85㎡ 초과 평형별 거래 건수와 평균 금액
- 두 번째 적재 뒤에도 첫 번째 적재 시점의 테이블과 집계를 똑같이 다시 꺼내는 방법

## 1단계 · 준비 — 라이브러리 · Java · 데이터

이 프로젝트는 세 가지 도구를 씁니다.

- **pyiceberg** — 파이썬에서 Iceberg 테이블을 만들고 읽고 씁니다. 카탈로그(테이블 목록)는 sqlite 파일 하나, 데이터 창고는 로컬 폴더로 둡니다. 3주차 18장에서는 Spark 와 MinIO 위에 Iceberg 를 올렸는데, 여기서는 Colab 에서도 서버 없이 돌도록 파일만으로 꾸립니다.
- **PySpark** — 5주차의 Spark 입니다. Java 가 있어야 돕니다.
- **Streamlit** — 7~8주차에 화면을 붙인 그 도구입니다.

Colab 에 없는 것만 설치합니다. 내 컴퓨터에 이미 있으면 설치를 건너뜁니다.

In [ ]:
import sys, subprocess, importlib.util, shutil, os

def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)

need("pyiceberg", "pyiceberg[sql-sqlite,pyarrow]")
need("pyspark")
need("streamlit")

if shutil.which("java") is None:                      # Colab 에 Java 가 없을 때만
    subprocess.run("apt-get install -y -qq openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

os.environ["PYSPARK_PYTHON"] = sys.executable          # Spark 일꾼도 이 노트북과 같은 파이썬을 쓰게
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"             # 내 컴퓨터 안에서만 — 네트워크 주소 경고도 사라집니다
print("java:", shutil.which("java") or os.environ.get("JAVA_HOME"))

데이터는 두 갈래입니다.

- **국토교통부 OpenAPI**(공공데이터포털 「국토교통부_아파트 매매 실거래가 자료」) — 인증키가 있어야 합니다. 활용신청은 자동승인이지만 키가 실제로 동작하기까지 몇 시간 걸리기도 합니다.
- **스냅샷** — 서울 열린데이터광장 「서울시 부동산 실거래가 정보」(OA-21275)에서 계약일이 2026년 6~9월인 거래를 내려받아 둔 파일입니다(2026-10-28 기준, 35,610건). 이 자료도 국토교통부 실거래가 공개시스템을 연계한 것이고, 공공누리 1유형(출처표시)이라 다시 나눠 줄 수 있습니다. 아파트 · 연립다세대 · 오피스텔 · 단독다가구가 섞여 있고, 글자 인코딩은 원본 그대로 CP949 입니다.

In [ ]:
import urllib.request
import pandas as pd

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

path = fetch("seoul_realestate_trade_202606_09.csv.gz")
print(path, round(os.path.getsize(path) / 1e6, 1), "MB")

> **확인 포인트** — `data/seoul_realestate_trade_202606_09.csv.gz 1.0 MB` 가 찍힙니다.

## 2단계 · 수집 — OpenAPI(키) 또는 스냅샷

국토교통부 API 는 **구(시군구 코드 5자리) 하나 × 계약월 하나**씩 묻습니다. 서울 25개 구의 넉 달치를 모으려면 100번 부릅니다.

| 요청 변수 | 뜻 | 예 |
|---|---|---|
| `serviceKey` | 공공데이터포털 인증키 | 마이페이지의 일반 인증키 |
| `LAWD_CD` | 법정동코드 10자리 중 앞 5자리(시군구) | `11680`(강남구) |
| `DEAL_YMD` | 계약년월 6자리 | `202609` |
| `pageNo` · `numOfRows` | 쪽 번호 · 한 쪽 건수 | `1` · `1000` |

응답은 XML 이고, 거래 한 건이 `<item>` 하나입니다. 금액 `dealAmount` 는 `"185,000"` 처럼 쉼표가 든 글자(만원)로 옵니다.

키는 코드에 쓰지 않습니다. Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `DATA_GO_KR_KEY` 로 넣어 두면 아래 셀이 읽습니다. 키가 없으면 수집을 건너뛰고 스냅샷으로 갑니다.

In [ ]:
import requests
import xml.etree.ElementTree as ET

KEY = os.environ.get("DATA_GO_KR_KEY")
try:
    from google.colab import userdata
    KEY = KEY or userdata.get("DATA_GO_KR_KEY")
except Exception:
    pass

URL = "https://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade"

def molit(lawd_cd, ym, key):
    """구 하나 × 계약월 하나의 아파트 매매 거래를 모두 받아 DataFrame 으로."""
    rows, page = [], 1
    while True:
        r = requests.get(URL, params={"serviceKey": key, "LAWD_CD": lawd_cd, "DEAL_YMD": ym,
                                      "pageNo": page, "numOfRows": 1000}, timeout=30)
        root = ET.fromstring(r.content)
        code = root.findtext(".//resultCode")
        if code not in ("00", "000"):
            raise RuntimeError(f"{code} {root.findtext('.//resultMsg')}")
        items = root.findall(".//item")
        rows += [{c.tag: (c.text or "").strip() for c in it} for it in items]
        if page * 1000 >= int(root.findtext(".//totalCount") or 0):
            return pd.DataFrame(rows)
        page += 1

if KEY:
    api = molit("11680", "202609", KEY)                       # 강남구 2026년 9월
    print(api.shape)
    print(api[["umdNm", "aptNm", "excluUseAr", "dealAmount", "dealDay"]].head())
else:
    print("인증키가 없습니다 — 스냅샷으로 진행합니다.")

키 없이 부르면 어떻게 되는지 일부러 한 번 봅니다. 공공데이터포털은 등록되지 않은 키를 HTTP 403 과 함께 XML 오류 메시지로 돌려줍니다.

In [ ]:
try:
    r = requests.get(URL, params={"serviceKey": "NOT-A-KEY", "LAWD_CD": "11680", "DEAL_YMD": "202609"}, timeout=15)
    print("HTTP", r.status_code, "→", ET.fromstring(r.content).findtext(".//returnAuthMsg"))
except Exception as e:
    print("요청 실패(네트워크):", type(e).__name__)

> **자주 겪는 오류** — 키를 주소창에 붙일 때 `+` · `/` · `=` 가 든 「인코딩 전」 키를 그대로 쓰면 `SERVICE_KEY_IS_NOT_REGISTERED_ERROR` 가 나기 쉽습니다.
> `requests` 의 `params=` 에는 **디코딩된 키**를 넣으세요. 라이브러리가 알아서 한 번 인코딩합니다.

이제 스냅샷을 읽습니다. 공공기관 CSV 라 인코딩을 알려 주어야 합니다(M1 2단계와 같습니다).

In [ ]:
raw = pd.read_csv(path, encoding="___①___", dtype=str)   # ① 한글 윈도 인코딩
print(raw.shape)
raw["건물용도"].value_counts()

> **확인 포인트** — `(35610, 21)` — 아파트 19,102건 · 연립다세대 11,720건 · 오피스텔 3,576건 · 단독다가구 1,212건입니다.

In [ ]:
assert raw.shape == (35610, 21)
assert raw["건물용도"].value_counts()["아파트"] == 19102

> **Aebon's Tip** — API 로 받은 표와 스냅샷은 열 이름이 다릅니다(`aptNm` ↔ `건물명`, `excluUseAr` ↔ `건물면적(㎡)`, `dealAmount` ↔ `물건금액(만원)`).
> 다음 단계에서 둘 다 같은 영어 이름으로 바꾸면 그 뒤의 코드는 어느 쪽에서 왔든 똑같이 돕니다.

## 3단계 · 정제 — 아파트만, 해제된 거래 빼기, 평형 나누기

쓸 열만 영어 이름으로 바꾸고 아파트만 남깁니다.

In [ ]:
cols = {
    "자치구명": "gu", "법정동명": "dong", "건물명": "apt", "계약일": "deal_date",
    "물건금액(만원)": "price", "건물면적(㎡)": "area", "층": "floor",
    "취소일": "cancel_date", "건축년도": "build_year", "건물용도": "usage",
}
df = raw.rename(columns=cols)[list(cols.values())]
apt = df[df["usage"] == "___②___"].copy()   # ② 건물용도 값
print("아파트:", len(apt))

`취소일` 이 있는 행은 계약했다가 **해제한 거래**입니다. 신고는 되었지만 실제로 성사되지 않았으니 시세 계산에서 빼야 합니다.
해제 신고는 계약보다 늦게 들어오므로, 어제 받은 자료에는 정상이던 거래가 오늘 해제로 바뀌기도 합니다.

In [ ]:
cancelled = apt["cancel_date"].___③___()   # ③ 값이 있는가
print("해제된 거래:", cancelled.sum())
apt = apt[~cancelled].drop(columns=["usage", "cancel_date"])
print("남은 거래:", len(apt))

> **확인 포인트** — 해제 533건을 빼고 **18,569건**이 남습니다.

In [ ]:
assert cancelled.sum() == 533 and len(apt) == 18569

형을 바꾸고, 대시보드에 쓸 열 세 개를 만듭니다.

- `deal_ym` — 계약 연월(`2026-09`). Iceberg 테이블을 이 값으로 나눠(파티션) 저장합니다.
- `size_band` — 평형 구간. 주택 공급 기준을 따라 전용 60㎡ 이하 / 60~85㎡ / 85㎡ 초과로 나눕니다.
- `ppy` — 평당가(만원). 거래금액 ÷ (전용면적 ÷ 3.3058).

In [ ]:
apt["deal_date"] = pd.to_datetime(apt["deal_date"], format="___④___").dt.date   # ④ 20260930 모양
apt["price"] = pd.to_numeric(apt["price"]).astype("int64")
apt["area"] = pd.to_numeric(apt["area"])
apt["floor"] = pd.to_numeric(apt["floor"]).astype("int32")
apt["build_year"] = pd.to_numeric(apt["build_year"]).astype("int32")

apt["deal_ym"] = pd.to_datetime(apt["deal_date"]).dt.strftime("%Y-%m")
apt["size_band"] = pd.___⑤___(apt["area"], bins=[0, 60, 85, 1000],   # ⑤ 구간 나누기
                          labels=["S(<=60)", "M(60-85)", "L(>85)"]).astype(str)
apt["ppy"] = (apt["price"] / (apt["area"] / 3.3058)).round(1)

print(apt["deal_ym"].value_counts().sort_index().to_dict())
apt.head(3)

> **확인 포인트** — 월별 건수 `{'2026-06': 5374, '2026-07': 5914, '2026-08': 3640, '2026-09': 3641}`.
> 8 · 9월이 적은 것은 거래가 줄어서이기도 하지만, 신고 기한(30일)이 남아 아직 덜 들어온 탓도 있습니다 — 자료는 내려받은 날 기준입니다.

In [ ]:
assert apt["deal_ym"].value_counts().sort_index().tolist() == [5374, 5914, 3640, 3641]
assert apt.isna().sum().sum() == 0

> **자주 겪는 오류** — `pd.cut` 의 구간은 기본이 「왼쪽 열림, 오른쪽 닫힘」입니다. `(60, 85]` 라서 정확히 85.0㎡ 는 M 에 들어갑니다.
> 「85㎡ 초과」라는 기준과 맞는지 경계값을 꼭 확인하세요.

## 4단계 · Iceberg 1차 적재 — 「어제」의 테이블

pyiceberg 의 `SqlCatalog` 는 테이블 목록을 sqlite 파일에, 데이터와 메타데이터를 창고 폴더에 둡니다(3주차 18장의 카탈로그와 같은 역할 — 21장).
1차 적재에서는 6~8월 거래만 넣습니다. 실제 운영이라면 어제 밤 배치가 넣은 분량입니다.

In [ ]:
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

WAREHOUSE = os.path.abspath("warehouse")
os.makedirs(WAREHOUSE, exist_ok=True)
catalog = SqlCatalog("lake", uri=f"sqlite:///{WAREHOUSE}/catalog.db", warehouse=f"file://{WAREHOUSE}")
catalog.create_namespace_if_not_exists("realestate")

first = apt[apt["deal_ym"] <= "2026-08"]
second = apt[apt["deal_ym"] == "2026-09"]
first_arrow = pa.Table.___⑥___(first, preserve_index=False)   # ⑥ pandas → Arrow
first_arrow.schema

Arrow 표의 스키마를 그대로 Iceberg 테이블 스키마로 씁니다. 다시 실행해도 되도록 같은 이름의 테이블이 있으면 지우고 만듭니다.
그리고 `deal_ym` 으로 파티션을 나눕니다. 월별로 폴더가 갈라져, 「9월만」 읽을 때 다른 달 파일은 열지도 않습니다.

In [ ]:
if catalog.table_exists("realestate.apt_trade"):
    catalog.drop_table("realestate.apt_trade")
trade = catalog.create_table("realestate.apt_trade", schema=first_arrow.schema)
with trade.update_spec() as spec:
    spec.___⑦___("deal_ym")   # ⑦ 값 그대로 파티션

trade.___⑧___(first_arrow)   # ⑧ 행 덧붙이기
snap_yesterday = trade.current_snapshot()
print("스냅샷:", snap_yesterday.snapshot_id, "· 행 수:", len(trade.scan().to_arrow()))

> **확인 포인트** — 행 수 **14,928** — 6 · 7 · 8월을 더한 값(5,374 + 5,914 + 3,640)입니다. 스냅샷 번호는 실행할 때마다 다릅니다.

In [ ]:
assert len(trade.scan().to_arrow()) == 14928
assert len(trade.snapshots()) == 1

창고 폴더에 무엇이 생겼는지 봅니다. `data/` 아래에 월별 파티션 폴더와 Parquet 파일이, `metadata/` 아래에 스냅샷을 기록한 파일이 있습니다.

In [ ]:
for root, dirs, files in sorted(os.walk(os.path.join(WAREHOUSE, "realestate"))):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), WAREHOUSE)[:90])

이번에는 일부러 **형이 다른 데이터**를 넣어 봅니다. 금액을 글자로 바꾼 표를 append 하면, Iceberg 는 테이블 스키마(`price: long`)와 맞지 않는다며 거절합니다.
CSV 를 그냥 폴더에 쌓을 때는 이런 검사가 없어서, 잘못된 파일이 섞인 것을 나중에야 알게 됩니다.

In [ ]:
bad = first.head(3).astype({"price": str})
try:
    trade.append(pa.Table.from_pandas(bad, preserve_index=False))
except ValueError as e:
    print("적재 거절:", str(e).splitlines()[0])
print("스냅샷 수 그대로:", len(trade.snapshots()))

## 5단계 · Spark 집계 — 구별 · 평형별 평당가

Spark 는 5주차에 썼던 그대로 `SparkSession` 을 열고 SQL 로 집계합니다.
Spark 가 Iceberg 를 직접 읽으려면 `iceberg-spark-runtime` jar 와 카탈로그 설정이 필요합니다. 여기서는 Colab 에서 확실히 돌도록 **pyiceberg 가 읽은 표를 Spark DataFrame 으로 넘기는** 방법을 씁니다.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[2]").appName("p1-apt")
         .config("spark.ui.enabled", "false")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def to_spark(iceberg_table, snapshot_id=None):
    """Iceberg 테이블(또는 그 스냅샷)을 읽어 Spark 의 trade 뷰로 등록합니다."""
    pdf = iceberg_table.scan(snapshot_id=snapshot_id).to_pandas()
    sdf = spark.___⑨___(pdf)   # ⑨ pandas → Spark
    sdf.createOrReplaceTempView("trade")
    return sdf.count()

print("Spark 로 넘긴 행:", to_spark(trade))

구마다 거래 건수와 평당가 **중앙값**을 구합니다. 몇십억짜리 거래 몇 건이 평균을 크게 끌어올리기 때문에 시세는 중앙값으로 봅니다.

In [ ]:
by_gu = spark.sql("""
    SELECT gu, count(*) AS n, round(___⑩___(ppy, 0.5)) AS median_ppy   -- ⑩ 정확한 백분위수 함수
    FROM trade
    GROUP BY gu
    ORDER BY median_ppy DESC
""").toPandas()
print(by_gu.head(3))
print(by_gu.tail(3))

> 확인 포인트(6~8월): 1위 **강남구 평당 12,466만원**(479건), 꼴찌 도봉구 2,520만원 — 다섯 배쯤 차이 납니다.

In [ ]:
assert len(by_gu) == 25
assert by_gu.iloc[0]["gu"] == "강남구" and by_gu.iloc[0]["median_ppy"] == 12466
assert by_gu.iloc[-1]["gu"] == "도봉구" and by_gu.iloc[-1]["median_ppy"] == 2520

평형별로도 봅니다.

In [ ]:
by_size = spark.sql("""
    SELECT size_band, count(*) AS n, round(avg(price)) AS avg_price, round(percentile(ppy, 0.5)) AS median_ppy
    FROM trade
    GROUP BY ___⑪___   -- ⑪ 평형 구간마다
    ORDER BY size_band
""").toPandas()
by_size

> 확인 포인트(6~8월): 60㎡ 이하가 6,686건으로 가장 많고, 85㎡ 초과는 1,999건입니다.
> 평균 금액은 큰 평형일수록 높지만, 평당가 중앙값은 오히려 작은 평형이 높습니다(4,675만원 > 4,081만원).

In [ ]:
assert by_size.set_index("size_band")["n"].to_dict() == {"L(>85)": 1999, "M(60-85)": 6243, "S(<=60)": 6686}

Spark SQL 은 없는 열 이름을 쓰면 실행 전에 막고, 비슷한 이름을 알려 줍니다.

In [ ]:
try:
    spark.sql("SELECT gu, avg(pric) FROM trade GROUP BY gu").show()
except Exception as e:
    print(type(e).__name__, "—", str(e).splitlines()[0][:110])

대시보드가 매번 원본 수만 건을 다시 집계하지 않도록, **구 × 월 × 평형** 집계를 마트 테이블로 저장해 둡니다. 마트는 매일 통째로 다시 계산해 덮어씁니다(`overwrite`) — 첫날만 빈 테이블이라 덧붙입니다.

In [ ]:
MART_SQL = """
    SELECT gu, deal_ym, size_band, count(*) AS n, round(percentile(ppy, 0.5), 1) AS median_ppy
    FROM trade
    GROUP BY gu, deal_ym, size_band
"""

def build_mart():
    m = spark.sql(MART_SQL).toPandas()
    m["n"] = m["n"].astype("int64")
    return pa.Table.from_pandas(m, preserve_index=False)

mart_arrow = build_mart()
if catalog.table_exists("realestate.apt_mart"):
    catalog.drop_table("realestate.apt_mart")
mart = catalog.create_table("realestate.apt_mart", schema=mart_arrow.schema)
mart.append(mart_arrow)                       # 첫날은 빈 테이블이라 덧붙이기
mart_yesterday = mart.current_snapshot()
print("마트 행 수:", len(mart.scan().to_arrow()))

> **확인 포인트** — 마트는 **225행**입니다(25개 구 × 3개월 × 3개 평형).

In [ ]:
assert len(mart.scan().to_arrow()) == 225

## 6단계 · 2차 적재 — 「오늘」 9월분이 들어온다

하루가 지나 9월 거래가 들어왔다고 생각하세요. 원본 테이블에 덧붙이고, 마트를 다시 계산해 덮어씁니다.

In [ ]:
import time
time.sleep(1)                                   # 두 스냅샷의 시각이 확실히 갈리게
trade.append(pa.Table.from_pandas(second, preserve_index=False))
snap_today = trade.current_snapshot()

to_spark(trade)
mart.___⑫___(build_mart())   # ⑫ 통째로 바꿔 쓰기
mart_today = mart.current_snapshot()

print("원본 행 수:", len(trade.scan().to_arrow()), "· 마트 행 수:", len(mart.scan().to_arrow()))
print(trade.inspect.snapshots().to_pandas()[["snapshot_id", "operation"]])
print(mart.inspect.snapshots().to_pandas()[["snapshot_id", "operation"]])

> **확인 포인트** — 원본 **18,569행**(14,928 + 3,641) · 마트 **300행**(25 × 4개월 × 3). 원본 스냅샷은 `append` 두 줄입니다.
> 마트는 세 줄입니다 — pyiceberg 의 `overwrite` 는 「옛 행 지우기(delete)」와 「새 행 쓰기(append)」를 스냅샷 두 개로 남기기 때문입니다. 가운데 스냅샷은 마트가 잠깐 비어 있던 순간입니다.

In [ ]:
assert len(trade.scan().to_arrow()) == 18569 and len(mart.scan().to_arrow()) == 300
assert [s.snapshot_id for s in trade.snapshots()] == [snap_yesterday.snapshot_id, snap_today.snapshot_id]
assert [s.summary.operation.value for s in mart.snapshots()] == ["append", "delete", "append"]

파티션별 행 수도 Iceberg 메타데이터에서 바로 볼 수 있습니다. 데이터 파일을 열지 않고 메타데이터만 읽으므로 빠릅니다.

In [ ]:
trade.inspect.___⑬___().to_pandas()[["partition", "record_count", "file_count"]]   # ⑬ 파티션별 요약

## 7단계 · 타임트래블 — 「어제 기준」으로 다시 보기

스냅샷 번호를 주면 그때의 테이블을 그대로 읽습니다. 오늘 9월분이 들어온 뒤에도 어제 테이블에는 9월이 없습니다.

In [ ]:
old = trade.scan(snapshot_id=snap_yesterday.___⑭___).to_pandas()   # ⑭ 스냅샷 번호
now = trade.scan().to_pandas()
print("어제:", len(old), sorted(old["deal_ym"].unique()))
print("오늘:", len(now), sorted(now["deal_ym"].unique()))

> **확인 포인트** — 어제 14,928행(6~8월) · 오늘 18,569행(6~9월).

In [ ]:
assert len(old) == 14928 and "2026-09" not in set(old["deal_ym"])
assert len(now) == 18569

번호를 몰라도 **시각**으로 찾을 수 있습니다. 「어제 적재가 끝난 직후」의 시각을 주면 그 시각에 살아 있던 스냅샷을 돌려줍니다.

In [ ]:
t_between = snap_yesterday.timestamp_ms + 500         # 1차 적재 0.5초 뒤 — 「어제 밤」
asof = trade.___⑮___(t_between)   # ⑮ 시각으로 스냅샷 찾기
print(asof.snapshot_id == snap_yesterday.snapshot_id)
assert asof.snapshot_id == snap_yesterday.snapshot_id

마트에도 같은 일을 합니다. 어제 대시보드에 걸려 있던 강남구 숫자와 오늘 숫자를 나란히 놓습니다.

In [ ]:
def gangnam(snapshot_id):
    m = mart.scan(snapshot_id=snapshot_id).to_pandas()
    g = m[(m["gu"] == "강남구") & (m["size_band"] == "M(60-85)")]
    return g.sort_values("deal_ym").set_index("deal_ym")["median_ppy"]

pd.DataFrame({"yesterday": gangnam(mart_yesterday.snapshot_id), "today": gangnam(mart_today.snapshot_id)})

> **확인 포인트** — 6~8월 값은 어제와 오늘이 같고, 오늘에만 9월 줄이 있습니다(어제 칸은 NaN).
> 회의 자료의 숫자가 「어제 기준」이었다면, 그 스냅샷을 열어 똑같이 재현할 수 있습니다.

In [ ]:
cmp = pd.DataFrame({"yesterday": gangnam(mart_yesterday.snapshot_id), "today": gangnam(mart_today.snapshot_id)})
assert cmp.loc[["2026-06", "2026-07", "2026-08"]].eval("yesterday == today").all()
assert pd.isna(cmp.loc["2026-09", "yesterday"])

없는 스냅샷 번호로 읽으면 어떻게 될까요? 오래된 스냅샷을 정리(expire)하고 나면 이런 일이 실제로 생깁니다.

In [ ]:
try:
    trade.scan(snapshot_id=123).to_arrow()
except Exception as e:
    print(type(e).__name__, "—", e)

> **Aebon's Tip** — 스냅샷은 지우기 전까지 데이터 파일을 붙잡고 있습니다. 운영에서는 「7일 지난 스냅샷 정리」 같은 규칙을 둡니다 — 그만큼만 과거로 돌아갈 수 있다는 뜻입니다.

## 8단계 · 대시보드 — Streamlit 앱 만들기

대시보드는 마트 테이블만 읽습니다. 화면 위쪽 라디오 버튼으로 **어느 스냅샷 기준으로 볼지** 고르게 만듭니다.
`%%writefile` 로 앱 파일을 이 폴더에 씁니다.

In [ ]:
%%writefile app.py
import os
from datetime import datetime
import pandas as pd
import streamlit as st
from pyiceberg.catalog.sql import SqlCatalog

st.set_page_config(page_title="Seoul apartment trades", layout="wide")
WAREHOUSE = os.path.abspath("warehouse")
catalog = SqlCatalog("lake", uri=f"sqlite:///{WAREHOUSE}/catalog.db", warehouse=f"file://{WAREHOUSE}")
mart = catalog.load_table("realestate.apt_mart")

snaps = [s for s in mart.snapshots() if s.summary.operation.value == "append"]   # 지우기만 한 스냅샷은 건너뜀
labels = [f"#{i + 1} {datetime.fromtimestamp(s.timestamp_ms / 1000):%Y-%m-%d %H:%M:%S}" for i, s in enumerate(snaps)]
pick = st.radio("기준 스냅샷", labels, index=len(labels) - 1, horizontal=True)
df = mart.scan(snapshot_id=snaps[labels.index(pick)].snapshot_id).to_pandas()

st.title("서울 아파트 매매 — 평당가(만원/3.3㎡)")
size = st.selectbox("평형", sorted(df["size_band"].unique()), index=1)
gus = st.multiselect("구", sorted(df["gu"].unique()), default=["강남구", "서초구", "노원구", "도봉구"])
view = df[(df["size_band"] == size) & (df["gu"].isin(gus))]

c1, c2 = st.columns(2)
c1.metric("거래 건수", f"{int(view['n'].sum()):,}")
c2.metric("기간", f"{df['deal_ym'].min()} ~ {df['deal_ym'].max()}")
st.line_chart(view.pivot(index="deal_ym", columns="gu", values="median_ppy"))
st.bar_chart(df[df["size_band"] == size].groupby("gu")["n"].sum().sort_values(ascending=False))

Streamlit 은 웹 서버라서 Colab 노트북 안에서는 화면을 바로 띄우기 어렵습니다. 대신 **서버 없이 앱 스크립트를 한 번 돌려 오류가 없는지** 검사합니다. `AppTest` 는 Streamlit 이 제공하는 시험 도구입니다.

In [ ]:
import logging
from streamlit.testing.v1 import AppTest
logging.getLogger("streamlit").setLevel(logging.ERROR)     # 서버 없이 돌릴 때 나오는 경고 끄기

at = AppTest.from_file(os.path.abspath("app.py"), default_timeout=60).run()
print("오류:", list(at.exception))
print("지표:", [(m.label, m.value) for m in at.metric])
print("스냅샷 선택지:", len(at.radio[0].options))

> **확인 포인트** — 오류가 없고, 지표는 `거래 건수 1,612`(60~85㎡, 기본으로 고른 4개 구) · `기간 2026-06 ~ 2026-09`, 스냅샷 선택지는 2개입니다.

In [ ]:
assert len(at.exception) == 0
assert at.metric[0].value == "1,612" and len(at.radio[0].options) == 2

라디오에서 첫 번째(어제) 스냅샷을 고르면 기간이 8월에서 끝나는지 확인합니다.

In [ ]:
at.radio[0].set_value(at.radio[0].options[0]).run()
print("어제 기준 기간:", at.metric[1].value)
assert at.metric[1].value == "2026-06 ~ 2026-08"

내 컴퓨터에서 화면으로 보려면 이렇게 하세요.

```bash
# 1) 이 노트북을 내려받아 내 컴퓨터의 Jupyter(VS Code)에서 끝까지 실행 → warehouse/ 와 app.py 가 생깁니다
# 2) 같은 폴더의 터미널에서
pip install streamlit "pyiceberg[sql-sqlite,pyarrow]"
streamlit run app.py          # 브라우저에서 http://localhost:8501
```

> **자주 겪는 오류** — Colab 에서 만든 `warehouse/` 폴더만 내려받아 내 컴퓨터에서 열면 `FileNotFoundError` 가 납니다.
> Iceberg 메타데이터에는 데이터 파일의 **절대 경로**(`file:///content/warehouse/...`)가 적혀 있기 때문입니다. 창고는 만든 곳에서 열거나, 처음부터 MinIO(17 · 19장) 같은 공용 저장소에 두세요.

노트북 안에서도 같은 그림을 matplotlib 으로 그려 둡니다(보고서에 붙일 그림). Streamlit 을 불러오면 matplotlib 이 그림을 화면에 내지 않는 방식으로 바뀌므로, 첫 줄 `%matplotlib inline` 으로 되돌립니다.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

m = mart.scan().to_pandas()
m = m[m["size_band"] == "M(60-85)"]
top = m.groupby("gu")["n"].sum().sort_values(ascending=False).head(6).index
eng = {"강남구": "Gangnam", "강동구": "Gangdong", "강북구": "Gangbuk", "강서구": "Gangseo", "관악구": "Gwanak",
       "광진구": "Gwangjin", "구로구": "Guro", "금천구": "Geumcheon", "노원구": "Nowon", "도봉구": "Dobong",
       "동대문구": "Dongdaemun", "동작구": "Dongjak", "마포구": "Mapo", "서대문구": "Seodaemun", "서초구": "Seocho",
       "성동구": "Seongdong", "성북구": "Seongbuk", "송파구": "Songpa", "양천구": "Yangcheon", "영등포구": "Yeongdeungpo",
       "용산구": "Yongsan", "은평구": "Eunpyeong", "종로구": "Jongno", "중구": "Jung", "중랑구": "Jungnang"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for g in top:
    s = m[m["gu"] == g].sort_values("deal_ym")
    ax1.plot(s["deal_ym"], s["median_ppy"], marker="o", label=eng[g])
ax1.set_title("Median price per pyeong, 60-85 m2 (10k KRW)")
ax1.set_xlabel("contract month")
ax1.legend(fontsize=8)
ax1.grid(alpha=.3)

g25 = by_gu.sort_values("median_ppy")
ax2.barh(range(len(g25)), g25["median_ppy"])
ax2.set_yticks(range(len(g25)))
ax2.set_yticklabels([eng[g] for g in g25["gu"]], fontsize=7)
ax2.set_title("Median price per pyeong by gu, Jun-Aug (10k KRW)")
plt.tight_layout()
plt.show()

> **확인 포인트** — 왼쪽은 거래가 많은 6개 구의 월별 추이, 오른쪽은 25개 구 막대입니다. 맨 위 막대가 강남(Gangnam)입니다.

In [ ]:
spark.stop()

## 9단계 · 더 해 보기

- 키를 받았다면 2단계 `molit()` 로 서울 25개 구 × 넉 달을 직접 모아 보세요. 시군구 코드 목록은 스냅샷의 `자치구코드` 열에 있습니다. 스냅샷과 건수가 같은지 비교하고, 다르면 왜 다른지(수집 날짜, 해제 신고) 적어 보세요.
- 해제 신고는 늦게 들어옵니다. 다음 날 같은 달을 다시 받으면 어떤 거래가 해제로 바뀌는지 — Iceberg 의 `overwrite`(필터를 준 부분 덮어쓰기)나 `delete` 로 처리해 보고, 스냅샷으로 「바뀌기 전」을 확인하세요.
- 5주차에 쓴 `iceberg-spark-runtime-3.5_2.12` jar 를 `spark.jars.packages` 로 붙이면 Spark 가 Iceberg 를 직접 읽습니다. `SELECT * FROM lake.realestate.apt_trade VERSION AS OF <스냅샷 번호>` 처럼 SQL 로 타임트래블할 수 있습니다(카탈로그를 JDBC 로 설정해야 같은 sqlite 를 봅니다).
- 대시보드에 「건축년도 구간」(2000년 이전 / 이후) 필터를 붙여, 같은 구 안에서도 신축과 구축의 평당가가 얼마나 다른지 보세요.

## 정답

① `cp949` — 한글 윈도 인코딩
② `아파트` — 건물용도 값
③ `notna` — 값이 있는가
④ `%Y%m%d` — 20260930 모양
⑤ `cut` — 구간 나누기
⑥ `from_pandas` — pandas → Arrow
⑦ `add_identity` — 값 그대로 파티션
⑧ `append` — 행 덧붙이기
⑨ `createDataFrame` — pandas → Spark
⑩ `percentile` — 정확한 백분위수 함수
⑪ `size_band` — 평형 구간마다
⑫ `overwrite` — 통째로 바꿔 쓰기
⑬ `partitions` — 파티션별 요약
⑭ `snapshot_id` — 스냅샷 번호
⑮ `snapshot_as_of_timestamp` — 시각으로 스냅샷 찾기